# Part 4 · Daily underperforming player lists

Every eligible player is checked as of a score date; one primary list per player by precedence: **Technical or payment friction → Revenue decline → Engagement decline**. Logic lives in `src/sciplay/lists/`.

This first part sets the thresholds and priority tiers. **All thresholds are starting values** in `src/sciplay/lists/config.py` and can be tuned.

**Contents**
1. Eligibility over time
2. Signals on one date, before precedence
3. Threshold sweep: list sizes and stability
4. Backtest: do the signals point at players who then play or spend less?
5. Opportunity gap and priority tiers
6. The daily flow and output contract
7. Two consecutive dates: 3 and 4 Nov
8. One player who changes list and priority
9. History: entry, exit, duration

In [ ]:
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display

from sciplay.lists import analysis as A, config as L
from sciplay.ltv import train as T

pio.renderers.default = 'plotly_mimetype+notebook_connected'
pd.set_option('display.width', 200)
data = T.load_data()
dates = pd.date_range('2026-10-09', '2026-11-04')          # days with LTV scores
runs = A.run_dates(data, dates)
print(f'{len(dates)} score dates, {dates[0]:%d %b} – {dates[-1]:%d %b}')

## 1. Eligibility over time

Eligible = not a cheater or legal settlement, at least 3 days old, seen in the last 30 days (Part 1).

In [ ]:
sz = A.sizes(runs)
fig = go.Figure()
for name, color in zip(L.PRECEDENCE, ['#E8664A', '#6B3FA0', '#3A86C8']):
    fig.add_bar(x=sz['date'], y=sz[name], name=name, marker_color=color)
fig.update_layout(barmode='stack', title='Players on each list per day (one primary list each)', template='plotly_white',
                  height=380, legend=dict(orientation='h', y=-0.2))
fig.show()
sz.describe().loc[['mean', 'min', 'max']].round(1)

,eligible,Technical or payment friction,Revenue decline,Engagement decline
mean,158.6,7.9,5.8,25.9
min,150.0,4.0,3.0,21.0
max,187.0,12.0,9.0,31.0


## 2. Signals on one date, before precedence

How many eligible players trigger each list on 4 Nov, and how much the lists overlap.

In [ ]:
s = runs[pd.Timestamp('2026-11-04')]
e = s[s.eligible]
print('Eligible:', len(e), '| ineligible:', s.loc[~s.eligible, 'ineligible_reason'].value_counts().to_dict())
display(pd.DataFrame({'friction': e.any_friction, 'revenue': e.any_revenue, 'engagement': e.any_engagement})
        .value_counts().rename('players').to_frame())
print('Valuable:', e.valuable.replace('', 'not valuable').value_counts().to_dict())

Eligible: 187 | ineligible: {'not seen in 30 days (churned)': 440, 'under 3 days old': 73}


players
friction revenue engagement         
False    False   False           144
                 True             24
True     False   False            11
False    True    True              6
                 False             2

Valuable: {'not valuable': 168, 'payer': 10, 'high predicted value': 9}


## 3. Threshold sweep: list sizes and stability

Each threshold varied on its own, all other values at their defaults, over the 27 dates. *New on list vs previous day* = share of a day's listed players who were not listed the day before (lower = more stable lists).

In [ ]:
grid = {'ACTIVITY_DROP': [0.3, 0.4, 0.5, 0.6, 0.7], 'SPEND_DROP': [0.3, 0.5, 0.7],
        'RECENT_DAYS': [7, 14], 'OVERDUE_FACTOR': [1.5, 2.0, 3.0]}
A.sweep(data, dates, grid)

,parameter,value,default,avg Technical or payment friction,avg Revenue decline,avg Engagement decline,new on list vs previous day
0,ACTIVITY_DROP,0.3,False,7.9,5.8,23.1,0.10
1,ACTIVITY_DROP,0.4,False,7.9,5.8,24.6,0.09
2,ACTIVITY_DROP,0.5,True,7.9,5.8,25.9,0.09
3,ACTIVITY_DROP,0.6,False,7.9,5.8,27.8,0.09
4,ACTIVITY_DROP,0.7,False,7.9,5.8,29.4,0.09
5,SPEND_DROP,0.3,False,7.9,5.9,25.9,0.09
6,SPEND_DROP,0.5,True,7.9,5.8,25.9,0.09
7,SPEND_DROP,0.7,False,7.9,5.4,25.9,0.09
8,RECENT_DAYS,7.0,True,7.9,5.8,25.9,0.09
9,RECENT_DAYS,14.0,False,15.6,5.6,22.9,0.07


Read with the backtest below: a looser threshold mostly adds players to the engagement list (part of that list comes from the return signal: away 7–21 days, longer than usual for the player); the friction and revenue lists are small at any setting because few players have friction events or have ever paid.

## 4. Backtest: do the signals point at players who then play or spend less?

At weekly dates from 1 Aug to 5 Oct (the next 30 days are fully in the data), compare flagged with not-flagged **comparable** eligible players: share who **stopped playing** (no day played in the next 30), days played in the next 30, and payers' spend in the next 30 days ÷ their last 28. Comparable = had a meaningful baseline for that signal: engagement — played at least weekly before (or young with active peers); friction — played 3+ of the last 7 days (heavy players crash more simply because they play more); revenue — payers and high-predicted-value players. Each signal on its own (before precedence). Small numbers: directional.

In [ ]:
A.backtest(data, pd.date_range('2026-08-01', '2026-10-05', freq='7D'))

dates  player_days  stopped_playing  days_played_next_30  payer_spend_next_vs_last
signal                    group                                                                                                      
Engagement decline signal flagged                     10          167             0.41                 4.47                      0.45
                          not flagged, comparable     10          393             0.27                10.75                      0.58
Friction signal           flagged                     10           50             0.05                18.35                      0.74
                          not flagged, comparable     10          232             0.06                14.83                      0.53
Revenue decline signal    flagged                     10           44             0.23                 6.39                      0.45
                          not flagged, comparable     10           54             0.32                 9.44                      0.59

## 5. Opportunity gap and priority tiers

Opportunity gap = expected value at the player's baseline − revised value now, never below $0 (older players: update-rule LTV at the previous vs the recent spending pace; model players: dollars behind forecast × 2.49). Distribution over all listed player-days:

In [ ]:
g = A.gap_distribution(runs)
display(g.groupby('list_type').opportunity_gap.describe(percentiles=[.5, .75, .9]).round(2))
for hi, mid in [(10, 1), (20, 2), (50, 5)]:
    print(f'high >= ${hi}, medium >= ${mid}:', (g.opportunity_gap >= hi).mean().round(2), 'high,',
          ((g.opportunity_gap >= mid) & (g.opportunity_gap < hi)).mean().round(2), 'medium')

,count,mean,std,min,50%,75%,90%,max
list_type,,,,,,,,
Engagement decline,699.0,0.00,0.03,0.0,0.00,0.00,0.00,0.60
Revenue decline,156.0,75.95,155.64,0.0,11.88,41.45,329.06,585.30
Technical or payment friction,213.0,0.07,0.44,0.0,0.00,0.00,0.00,3.43


high >= $10, medium >= $1: 0.08 high, 0.01 medium
high >= $20, medium >= $2: 0.06 high, 0.03 medium
high >= $50, medium >= $5: 0.04 high, 0.05 medium


**Decision (b): dollars first, severity second.** High = opportunity gap ≥ $20; Medium = gap ≥ $2, or money directly blocked (failed or incomplete purchases), or a severe decline (score ≥ 75); Low otherwise. Dollar tiers alone would make every friction and engagement player low, every day: they are non-payers with no dollar gap.

## 6. The daily flow and output contract

`python -m sciplay.lists.run --date 2026-11-04` (or `--from … --to …`):
1. **No rows for the date** (after the data end): nothing listed, the run is logged as `no_data`.
2. **Health check for the date fails** (late or incomplete rows): the list is still written, flagged `data_incomplete`.
3. LTV scores for the date are read, or computed and stored if missing.
4. Signals as of the date (rows dated on or before it only), one primary list by precedence, score, priority, reasons, action.
5. Stored in SQLite `daily_lists`, key (score date, player, model + rules version): a rerun replaces rows; `list_runs` logs status, list sizes and ineligible players by reason.

**Underperformance score (0–100):** 0 = at the player's baseline; 100 = spend or play stopped, or severe friction. Friction: 25 points per event in the last 7 days. Revenue: the largest of spend drop %, overdue, % behind forecast, activity drop % for high-value non-payers. Engagement: the largest drop in days played or sessions, or time away.

In [ ]:
from sciplay.lists import history as H, run as R
runs_log = R.read(None, table=R.RUNS)
runs_log[['score_date', 'status', 'eligible', 'listed', 'list_sizes', 'ineligible']].tail(5)

,score_date,status,eligible,listed,list_sizes,ineligible
77,2026-10-31,ok,168,44,"{""Engagement decline"": 28, ""Technical or payme...","{""not seen in 30 days (churned)"": 425, ""under ..."
78,2026-11-01,ok,170,44,"{""Engagement decline"": 28, ""Technical or payme...","{""not seen in 30 days (churned)"": 427, ""under ..."
79,2026-11-02,ok,166,44,"{""Engagement decline"": 27, ""Technical or payme...","{""not seen in 30 days (churned)"": 432, ""under ..."
80,2026-11-03,ok,164,42,"{""Engagement decline"": 25, ""Revenue decline"": ...","{""not seen in 30 days (churned)"": 436, ""under ..."
81,2026-11-04,ok,187,43,"{""Engagement decline"": 24, ""Technical or payme...","{""not seen in 30 days (churned)"": 440, ""under ..."


## 7. Two consecutive dates: 3 and 4 Nov (the delivery simulation uses the same days)

In [ ]:
for D in ['2026-11-03', '2026-11-04']:
    l = R.read(D)
    print(D, '-', len(l), 'players listed')
    display(l.groupby(['list_type', 'priority']).size().unstack(fill_value=0)
            .reindex(L.PRECEDENCE).reindex(columns=['high', 'medium', 'low'], fill_value=0))
R.read('2026-11-04')[['player_id', 'list_type', 'priority', 'predicted_ltv_365', 'underperformance_score',
                     'ltv_to_date', 'opportunity_gap', 'reason_text', 'recommended_action']].head(10).round(2)

2026-11-03 - 42 players listed


priority,high,medium,low
list_type,,,
Technical or payment friction,0,7,1
Revenue decline,4,4,1
Engagement decline,0,22,3


2026-11-04 - 43 players listed


priority,high,medium,low
list_type,,,
Technical or payment friction,0,8,3
Revenue decline,3,5,0
Engagement decline,0,20,4


,player_id,list_type,priority,predicted_ltv_365,underperformance_score,ltv_to_date,opportunity_gap,reason_text,recommended_action
1,1792,Engagement decline,low,0.00,56.0,0.00,0.00,Played 2 of the last 7 days vs 4.5 a week before,CRM team: re-engagement push or email with fre...
3,13280,Engagement decline,medium,0.00,100.0,0.00,0.00,Not seen for 17 days; usually plays every day,CRM team: come-back message with a free-coins ...
6,33840,Revenue decline,high,2713.83,100.0,2366.73,266.79,No purchase for 9 days; usually every 1.5 days...,VIP team: timed offer on the player's usual pu...
9,40013,Revenue decline,medium,12.98,100.0,12.98,0.00,No purchase for 97 days; usually every 5.5 day...,VIP team: timed offer on the player's usual pu...
12,50019,Engagement decline,medium,0.00,100.0,0.00,0.00,0 sessions in the last 7 days vs 2.2 a week be...,CRM team: re-engagement push or email with fre...
15,59937,Technical or payment friction,medium,86.05,50.0,37.31,0.00,"2 purchase attempts in the last 7 days, none c...",Player support team (payments): check store an...
18,63229,Engagement decline,medium,0.00,100.0,0.00,0.00,0 sessions in the last 7 days vs 3 a week befo...,CRM team: re-engagement push or email with fre...
21,64012,Engagement decline,medium,0.00,100.0,0.00,0.00,Played 0 of the last 7 days vs 1 a week before...,CRM team: re-engagement push or email with fre...
24,73769,Revenue decline,medium,172.01,100.0,159.20,0.00,No purchase for 20 days; usually every 2 days ...,VIP team: timed offer on the player's usual pu...
26,84196,Engagement decline,medium,0.00,100.0,0.00,0.00,Not seen for 9 days; usually plays every 2 days,CRM team: come-back message with a free-coins ...


In [ ]:
print('Changes on 4 Nov vs 3 Nov:')
H.changes('2026-11-04')

Changes on 4 Nov vs 3 Nov:


,player_id,change,before,after
0,105068,priority down,medium,low
1,109513,exited,Technical or payment friction (medium),
2,134687,exited,Technical or payment friction (medium),
3,159464,entered,,Engagement decline (low)
4,192447,exited,Engagement decline (low),
5,209049,entered,,Technical or payment friction (low)
6,224283,moved list,Revenue decline (high),Engagement decline (low)
7,229507,priority up,low,medium
8,254258,exited,Engagement decline (medium),
9,255517,exited,Engagement decline (medium),


## 8. One player who changes list and priority: player 224283

Installed 7 Sep; one purchase of $14.46 on 19 Sep; played almost daily until 9 Oct, then nothing until 3 Nov; bought again ($14.46) on 4 Nov.

In [ ]:
h = H.history()
p = h[h.player_id == 224283][['score_date', 'list_type', 'priority', 'underperformance_score', 'opportunity_gap', 'reason_text']]
p[p.score_date.isin(['2026-10-09', '2026-10-15', '2026-10-17', '2026-10-29', '2026-11-03', '2026-11-04'])]

,score_date,list_type,priority,underperformance_score,opportunity_gap,reason_text
2015,2026-10-09,Technical or payment friction,low,25.0,0.00,1 support ticket in the last 7 days (more than...
2248,2026-10-15,Engagement decline,medium,93.0,0.00,Played 1 of the last 7 days vs 5.2 a week befo...
2324,2026-10-17,Revenue decline,high,100.0,23.14,Spent $0.00 in the last 28 days vs $14.46 in t...
2782,2026-10-29,Revenue decline,high,100.0,23.12,Spent $0.00 in the last 28 days vs $14.46 in t...
3002,2026-11-03,Revenue decline,high,100.0,23.11,Spent $0.00 in the last 28 days vs $14.46 in t...
3072,2026-11-04,Engagement decline,low,65.0,0.00,6 sessions in the last 7 days vs 17.2 a week b...


- **9–14 Oct · Technical or payment friction (low):** a support ticket in the last 7 days. Friction comes first even though play is already falling.
- **15–16 Oct · Engagement decline (medium):** the ticket is older than 7 days; played 1, then 0 of the last 7 days against about 5 a week before (score 93 → 100, severe → medium).
- **17 Oct – 3 Nov · Revenue decline (high):** the purchase dropped out of the last 28 days: $0 vs $14.46 in the 28 before. At that pace the update rule's value falls by about $23 → high priority. Action: CRM/VIP offer.
- **4 Nov · Engagement decline (low):** the player came back and bought again, so the revenue decline ends; sessions are still well below their usual pace (6 vs 17 a week), so they stay on a list, at low priority.

This is the precedence rule at work: one list at a time, moving as the evidence changes.

## 9. History: entry, exit, duration

Each continuous stay on a list (27 run dates, 9 Oct – 4 Nov):

In [ ]:
sp = H.spells()
display(sp.groupby('list_type').agg(stays=('days', 'size'), median_days=('days', 'median'), longest=('days', 'max')))
sp.sort_values('days', ascending=False).head(8)

,stays,median_days,longest
list_type,,,
Engagement decline,85,5.0,27
Revenue decline,16,7.0,27
Technical or payment friction,43,4.0,18


,player_id,list_type,entered,last_seen_on_list,days,priorities
11,40013,Revenue decline,2026-10-09,2026-11-04,27,medium
85,212431,Engagement decline,2026-10-09,2026-11-04,27,low → medium
21,64012,Engagement decline,2026-10-09,2026-11-04,27,medium → low
23,73769,Revenue decline,2026-10-09,2026-11-04,27,high → medium
55,156487,Engagement decline,2026-10-09,2026-11-02,25,medium → low
71,192648,Engagement decline,2026-10-10,2026-11-02,24,low → medium
17,53842,Engagement decline,2026-10-09,2026-11-01,24,low → medium
112,247806,Engagement decline,2026-10-14,2026-11-04,22,low → medium
